In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.aml_bronze.raw_transactions")

# Ορίζουμε "κανόνες ποιότητας" που ΠΡΕΠΕΙ να ισχύουν πριν προχωρήσουμε
null_accounts = df.filter(
    F.col("from_account").isNull() | F.col("to_account").isNull()
).count()

negative_amounts = df.filter(F.col("amount_paid") < 0).count()

null_laundering_label = df.filter(F.col("is_laundering").isNull()).count()

print(f"Null accounts: {null_accounts}")
print(f"Negative amounts: {negative_amounts}")
print(f"Null laundering label: {null_laundering_label}")

In [0]:
validation_errors = []

if null_accounts > 0:
    validation_errors.append(f"Found {null_accounts} rows with null account IDs")

if negative_amounts > 0:
    validation_errors.append(f"Found {negative_amounts} rows with negative amounts")

if null_laundering_label > 0:
    validation_errors.append(f"Found {null_laundering_label} rows with null laundering label")

if validation_errors:
    error_message = "Data quality validation failed:\n" + "\n".join(validation_errors)
    raise ValueError(error_message)
else:
    print("✓ All validation checks passed. Proceeding to next stage.")

In [0]:
# Προσομοίωση: πάρε ένα μικρό sample και "χαλάστο" τεχνητά
sample = df.limit(1000)

corrupted = (
    sample
    .withColumn(
        "amount_paid",
        F.when(F.col("amount_paid") > 100000, F.lit(-999))  # τεχνητά "χαλασμένο" value
         .otherwise(F.col("amount_paid"))
    )
)

# Το πραγματικό dead-letter pattern:
good_rows = corrupted.filter(F.col("amount_paid") >= 0)
bad_rows = corrupted.filter(F.col("amount_paid") < 0)

print(f"Good rows: {good_rows.count()}")
print(f"Bad rows (sent to dead-letter): {bad_rows.count()}")

# Τα καλά προχωράνε κανονικά
good_rows.write.format("delta").mode("overwrite").saveAsTable("workspace.aml_bronze.validated_transactions")

# Τα κακά πάνε σε ξεχωριστό table, με metadata για το γιατί απορρίφθηκαν
(
    bad_rows
    .withColumn("_rejection_reason", F.lit("negative_amount"))
    .withColumn("_rejected_at", F.current_timestamp())
    .write.format("delta").mode("append")
    .saveAsTable("workspace.aml_bronze.rejected_transactions")
)

In [0]:
import time

def write_with_retry(dataframe, table_name, max_retries=3, delay_seconds=5):
    attempt = 0
    while attempt < max_retries:
        try:
            (
                dataframe.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(table_name)
            )
            print(f"✓ Successfully wrote to {table_name} on attempt {attempt + 1}")
            return True
        except Exception as e:
            attempt += 1
            print(f"✗ Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                print(f"Retrying in {delay_seconds} seconds...")
                time.sleep(delay_seconds)
            else:
                print(f"All {max_retries} attempts failed. Raising exception.")
                raise

# Δοκιμή σε πραγματικό table (θα πετύχει με την πρώτη)
write_with_retry(good_rows, "workspace.aml_bronze.validated_transactions")